In [2]:
import random
import math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
import time

import svg_canvas

In [3]:
# Constants
MEAN_POP = 3000
STD_DEV_POP = 20
GRID_SIZE = 32
MAX_ITERATIONS = 100
PROB_MOVE = 0.1

In [4]:
def init_grid(grid,distance, size):

    def inverse_sqr_distance(y1, x1, y2, x2):
        dx = x2 - x1
        dy = y2 - y1
        dist_sq = dx * dx + dy * dy
        if dist_sq == 0:
            return 0.0
        return 1.0 / dist_sq
    
    for row in range(size):
        grid.append([])
        distance.append([])
        for col in range(size):
            grid[row].append([]) 
            dist_row = []
            for y in range(size):
                for x in range(size):
                    d  = inverse_sqr_distance(row,col,y,x)
                    dist_row.append(d)
            distance[row].append(dist_row)

In [5]:
def init_population(population, grid):
    c_max = len(grid)
    r_max = len(grid[0])
    pop_count = 0 # count number in population
    for row in range(r_max):   #row loop
        for col in range(c_max):   # col loop
            act_pop = 0
            if row == int(r_max/2) and col == int(c_max/2):
                act_pop = int(random.gauss(MEAN_POP, STD_DEV_POP))
            for _ in range(act_pop): # person loop
                grid[row][col].append(pop_count)
                population.append([pop_count,row,col])
                pop_count = pop_count + 1

In [6]:
def move(row, col, grid, distance):

    def move_to(y,x):
        # if x!=5 or y!=5:
        #     if random.random()<0.1:
        #         return 5,5
        choice = random.random()
        for i in range(len(roulette_wheel)):
            if choice <= roulette_wheel[i]:
                return i // GRID_SIZE, i % GRID_SIZE
        return 0, 0  # Fallback (should not happen)
    
    # Calculate roulette wheel weights
    r_max = len(grid)
    c_max = len(grid[0])
    
    weights = distance[row][col]
    total_weight = sum(weights)
    
    roulette_wheel = []
    accumulate = 0.0
    for i in range(len(weights)):
        if total_weight > 0:
            accumulate = accumulate + (weights[i] / total_weight)
        roulette_wheel.append(accumulate)

    movers = []
    cell_pop = grid[row][col]
    for p in range(len(cell_pop)):
        if random.random()<PROB_MOVE:
            p_idx = cell_pop[p]
            nr, nc = move_to(row,col)
            movers.append([p_idx, nr, nc])
        
    return movers

In [7]:
def migrate(grid, population,distance):
    movers = []
    r_max = len(grid)
    c_max = len(grid[0])
    movers = []
    # Collect all movers
    for row in range(r_max):
        for col in range(c_max):
            movers+=move(row, col, grid,distance)
    
    # Execute moves
    for i in range(len(movers)):
        m = movers[i]
        p_id = m[0]  # This is the actual index in population array
        new_r = m[1]
        new_c = m[2]
        
        
        p = population[p_id]
        old_r = p[1]
        old_c = p[2]
        
        # Skip if moving to same cell
        if old_r == new_r and old_c == new_c: 
            continue
        
        # Update population record
        p[1] = new_r
        p[2] = new_c
        
        # Remove from old cell
        old_cell = grid[old_r][old_c]
        old_cell.remove(p_id)
        grid[new_r][new_c].append(p_id)
    
    return len(movers)
    

In [8]:
## Take this on faith
def plot_grid(grid):
    plt.imshow(grid, vmin=0.0, vmax=1.0)
    plt.colorbar()
    plt.show()

In [9]:
def draw_grid(svg,grid,population):
    r_max = len(grid)
    c_max = len(grid[0])
    for row in range(r_max):
        for col in range(c_max):
            pop = min(255,len(grid[row][col]))
            r_col = 255-pop
            if pop ==0:
                svg.addRect(row,col,f'rgb(200,220,200)')
            else:
                svg.addRect(row,col,f'rgb(255,{r_col},{r_col})')

In [13]:

iterations = widgets.IntSlider(value=50, min=1, max=200, description='Iterations:', style={'description_width': 'initial'})

# Layout them nicely
controls = widgets.VBox([
    iterations,
])
# Display everything
display(controls)

In [15]:

# Initialize data structures
grid = []
distance = []
population = []

# Initialize grid and population
init_grid(grid, distance, GRID_SIZE)
init_population(population, grid)

svg = svg_canvas.SVGCanvas(640,640,GRID_SIZE)
# 1. Create the output ONCE and give it a permanent ID
output_display = display(HTML("<div id='sim-container'>Loading...</div>"), display_id=True)
for t in range(iterations.value):
    migrate(grid, population,distance)
    svg.clear()
    draw_grid(svg,grid,population)
    new_svg = svg.getCanvas()
    new_svg += f'<p>{t}</p>'
    output_display.update(HTML(f"<div id='sim-container'>{new_svg}</div>"))
    time.sleep(0.1)

